# Histograms

In [ ]:
# datascience
from datascience import *
import numpy as np
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
plots.rcParams["patch.force_edgecolor"] = True


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


## Use counts and bar charts for categorical distributions

In [ ]:
# datascience
top_movies_2017 = Table.read_table('top_movies_2017.csv')
top_movies_2017.show(6)


In [ ]:
# polars
top_movies_2017_pl = pl.read_csv('top_movies_2017.csv', eol_char='\r')
top_movies_2017_pl.head(6)


In [ ]:
# TODO: visualize distribution of Studios

Exercise: Show the **percentage** of movies from each studio instead of the count

In [ ]:
# TODO: your answer here

## Use binning for numerical distributions

In [ ]:
# datascience
# Data from https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/
top_movies = Table.read_table('top_movies_2026.csv')
top_movies.show(4)


In [ ]:
# polars
# Data from https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/
top_movies_pl = pl.read_csv('top_movies_2026.csv')
top_movies_pl.head(4)


Optional: `datascience` [Formatters](https://data8.org/datascience/formats.html)

In [ ]:
# datascience
top_movies.set_format(3, CurrencyFormatter)


In [ ]:
# polars
top_movies_pl


In [ ]:
# TODO: explore data

In [ ]:
# TODO: what are the outliers?

### **Task**: Visualize the distribution of movie ages

In [ ]:
# datascience
ages = 2026 - top_movies.column('Year')
ages


In [ ]:
# polars
ages_pl = 2026 - top_movies_pl.get_column('Year')
ages_pl


In [ ]:
# datascience
top_movies = top_movies.with_column('Age', ages)
top_movies


In [ ]:
# polars
top_movies_pl = top_movies_pl.with_columns((ages_pl).alias('Age'))
top_movies_pl


In [ ]:
min(ages), max(ages)

- If you want to make equally sized bins, `np.arange()` is a great tool to help you.

In [ ]:
# datascience
np.arange(0, max(ages)+10, 10)


In [ ]:
# polars
pl.int_range(0, max(ages_pl)+10, 10, eager=True)


In [ ]:
# TODO: use bin method

In [ ]:
# datascience
top_movies.hist('Age', bins = np.arange(0, max(ages)+10, 10), unit = 'Year', normed=False)


In [ ]:
# polars
# hvplot
top_movies_pl.hvplot.hist(y='Age', normed=False, bins=(pl.int_range(0, max(ages_pl)+10, 10, eager=True)).to_list(), xlabel='Age (Year)')


---

- Otherwise, you can pick your own bins. These are just bins that we picked out.


In [ ]:
# datascience
my_bins = make_array(0, 5, 10, 15, 25, 60)


In [ ]:
# polars
my_bins_pl = pl.Series('values', [0, 5, 10, 15, 25, 60])


In [ ]:
# datascience
binned_data = top_movies.bin('Age', bins = my_bins)
binned_data


In [ ]:
# polars
binned_data_pl = top_movies_pl.get_column('Age').hist(bins=(my_bins_pl).to_list()).rename({'breakpoint': 'bin', 'count': 'Age count'}).drop('category')
binned_data_pl


**Note:** The last "bin" does not include any observations!!

## Introducing the histogram and the area principle

In [ ]:
# datascience
top_movies.hist('Age', bins = my_bins, unit = 'Year', normed=True)


In [ ]:
# polars
# hvplot
top_movies_pl.hvplot.hist(y='Age', normed=True, bins=(my_bins_pl).to_list(), xlabel='Age (Year)')


---

### **Discussion Question**: Compare the bins $[0, 5)$ and $[15, 25)$. 

- Which one has more movies?
- Which one is more crowded?

## Challenge tasks

### **Task**: Find the height of the $[15,25)$ bin in the histogram above.

$$\text{height} = \frac{\text{percent}}{\text{width}}$$

Add a column containing what percent of movies are in each bin (the **area** of each bin)

In [ ]:
# TODO

In [ ]:
# TODO

In [ ]:
# TODO

In [ ]:
# TODO

In [ ]:
# TODO

### **Task**: Find the heights of the (rest of the) bins.

$$\text{height} = \frac{\text{percent}}{\text{width}}$$

Remember that the last row in the table does not represent a bin!

In [ ]:
# datascience
height_table = binned_data.take(np.arange(binned_data.num_rows - 1))
height_table 


In [ ]:
# polars
height_table_pl = binned_data_pl.gather(pl.int_range(binned_data_pl.height - 1, eager=True))
height_table_pl 


Remember `np.diff`?

In [ ]:
# datascience
bin_widths = np.diff(binned_data.column('bin'))


In [ ]:
# polars
bin_widths_pl = (binned_data_pl.get_column('bin')).diff(null_behavior='drop')


In [ ]:
bin_widths

In [ ]:
# datascience
height_table = height_table.with_column('Width', bin_widths)
height_table


In [ ]:
# polars
height_table_pl = height_table_pl.with_columns((bin_widths_pl).alias('Width'))
height_table_pl


In [ ]:
# datascience
height_table = height_table.with_column('Height',
                                        height_table.column('Percent')/height_table.column('Width'))


In [ ]:
# polars
height_table_pl = height_table_pl.with_columns((height_table_pl.get_column('Percent')/height_table_pl.get_column('Width')).alias('Height'))


In [ ]:
height_table

To check our work one last time, let's see if the numbers in the last column match the heights of the histogram:

In [ ]:
# datascience
top_movies.hist('Age', bins = my_bins, unit = 'Year')


In [ ]:
# polars
# hvplot
top_movies_pl.hvplot.hist(y='Age', normed=True, bins=(my_bins_pl).to_list(), xlabel='Age (Year)')
